<a href="https://colab.research.google.com/github/debashistripathy-riku/LMS_celebal_weekly_Assignment/blob/main/customer_intelligence_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Intelligence & Country Segmentation System
### End-to-End Hybrid ML Pipeline (Clustering, Ensemble & Classification)

This self-contained notebook is designed to run in **Google Colab**. It loads the country socio-economic dataset directly from a public URL, performs preprocessing, clustering (K-Means & DBSCAN), and trains ensemble classifiers (Random Forest & XGBoost) on the discovered segments.

## 1. Setup and Libraries Installation
We install `xgboost` if it's not already present in the environment.

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, silhouette_score
from sklearn.preprocessing import LabelEncoder

# Style config
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load Dataset from Public URL

In [ ]:
dataset_url = "https://raw.githubusercontent.com/vatsalmandalia/Country-Data-Kmeans-Clustering/master/Country-data.csv"
df = pd.read_csv(dataset_url)

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

## 3. Exploratory Data Analysis (EDA)
Let's see some basic summary stats and correlations.

In [ ]:
df.describe()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Features")
plt.show()

## 4. Preprocessing & Feature Engineering
In the raw dataset, `exports`, `health`, and `imports` are given as a percentage of `gdpp`. We convert them to absolute per capita dollar values to avoid scale mismatches, then scale all variables.

In [ ]:
# 1. Convert columns to absolute values
df_proc = df.copy()
df_proc['exports'] = (df_proc['exports'] * df_proc['gdpp']) / 100.0
df_proc['health'] = (df_proc['health'] * df_proc['gdpp']) / 100.0
df_proc['imports'] = (df_proc['imports'] * df_proc['gdpp']) / 100.0

# 2. Extract numerical features
feature_cols = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
X = df_proc[feature_cols]

# 3. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("Scaled Features Preview:")
X_scaled_df.head()

### Dimensionality Reduction (PCA)
Reducing the features to 2 principal components to visualize clusters easily.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

## 5. Unsupervised Clustering

### 5.1 K-Means Clustering
We use Elbow method (inertia) and Silhouette Analysis to check $K=3$.

In [ ]:
wcss = []
sil_scores = []
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(2, 9), wcss, marker='o', color='purple')
ax1.set_title('Elbow Method')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')

ax2.plot(range(2, 9), sil_scores, marker='o', color='teal')
ax2.set_title('Silhouette Scores')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
plt.show()

### Fit K-Means ($K=3$) and Profiling
Let's map the clusters to logical segments based on mean GDP per capita values.

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
raw_labels = kmeans.fit_predict(X_scaled)

df_proc['kmeans_cluster'] = raw_labels

# Programmatic profiling map: sort by mean gdpp
cluster_profiles = df_proc.groupby('kmeans_cluster')['gdpp'].mean().sort_values().index
label_map = {cluster_profiles[0]: 'Underdeveloped',
             cluster_profiles[1]: 'Developing',
             cluster_profiles[2]: 'Developed'}

df_proc['development_segment'] = df_proc['kmeans_cluster'].map(label_map)
df_pca['segment'] = df_proc['development_segment']

print("Cluster Counts:")
print(df_proc['development_segment'].value_counts())
df_proc.groupby('development_segment')[['gdpp', 'child_mort', 'income']].mean()

In [ ]:
# Plot the K-Means clusters on PCA Projection
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='segment', palette='Set1', s=80, alpha=0.8)
plt.title('K-Means Clusters (PCA Projection)')
plt.show()

### 5.2 DBSCAN Clustering
Used to identify outliers that differ from the typical cluster density regions.

In [ ]:
dbscan = DBSCAN(eps=1.8, min_samples=3)
db_labels = dbscan.fit_predict(X_scaled)
df_proc['dbscan_cluster'] = db_labels
df_proc['dbscan_segment'] = ['Outlier' if l == -1 else f'Cluster_{l}' for l in db_labels]

print("DBSCAN Cluster counts:")
print(df_proc['dbscan_segment'].value_counts())

## 6. Supervised Ensemble Classification
We use the K-Means segments as pseudo-labels to train **Random Forest** and **XGBoost** classifiers.

In [ ]:
# Prepare data
le = LabelEncoder()
y = le.fit_transform(df_proc['development_segment'])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# 2. XGBoost
xgb = XGBClassifier(n_estimators=100, max_depth=3, eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

print("Random Forest Test Accuracy:", accuracy_score(y_test, rf_pred))
print("XGBoost Test Accuracy:", accuracy_score(y_test, xgb_pred))

In [ ]:
print("--- Random Forest Classification Report ---")
print(classification_report(y_test, rf_pred, target_names=le.classes_))

print("--- XGBoost Classification Report ---")
print(classification_report(y_test, xgb_pred, target_names=le.classes_))

In [ ]:
# Feature Importance Comparison
rf_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_}).sort_values(by='Importance', ascending=False)
xgb_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': xgb.feature_importances_}).sort_values(by='Importance', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=rf_imp, x='Importance', y='Feature', hue='Feature', palette='Blues_r', legend=False, ax=ax1)
ax1.set_title('Random Forest Feature Importance')

sns.barplot(data=xgb_imp, x='Importance', y='Feature', hue='Feature', palette='Oranges_r', legend=False, ax=ax2)
ax2.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

## 7. Real-Time Inference Console
Edit the variables below and run the cell to predict the segment of a new custom country.

In [ ]:
# Input custom country metrics below
custom_country = {
    'child_mort': 32.0,
    'exports': 20.0,    # as % of gdpp
    'health': 5.5,      # as % of gdpp
    'imports': 30.0,    # as % of gdpp
    'income': 18000,
    'inflation': 4.5,
    'life_expec': 72.0,
    'total_fer': 2.3,
    'gdpp': 12000
}

# 1. Convert percentage features to absolute values
cc_df = pd.DataFrame([custom_country])
cc_df['exports'] = (cc_df['exports'] * cc_df['gdpp']) / 100.0
cc_df['health'] = (cc_df['health'] * cc_df['gdpp']) / 100.0
cc_df['imports'] = (cc_df['imports'] * cc_df['gdpp']) / 100.0

# 2. Scale features
scaled_cc = scaler.transform(cc_df[feature_cols])

# 3. Predict segment
pred_class = xgb.predict(scaled_cc)
pred_label = le.inverse_transform(pred_class)[0]

print(f"Predicted Segment: {pred_label}")